In [ ]:
# ── Clone repo from GitHub (run once per Colab session) ───────────────────
import os, sys
if not os.path.exists('/content/protosearch'):
    !git clone https://github.com/bjdraper/protosearch.git /content/protosearch
sys.path.insert(0, '/content/protosearch')
os.chdir('/content/protosearch')
!pip install -q -r /content/protosearch/requirements.txt

# 03 — Trees & Ancestral State Reconstruction
1. MAFFT alignment + FastTree for each cluster
2. IQ-TREE2 ML tree with marginal ASR
3. Parse .state file → consensus ancestral sequences
4. Four diagnostic plots per cluster

In [ ]:
import os, sys
from pathlib import Path

# ── CONFIG (edit here) ────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/acyltransferase'
CLUSTERS      = ['lclat1', 'gpat4']   # which clusters to run trees + ASR for
RUN_IQTREE    = True                  # set False to skip ASR (fast tree only)
IQTREE_THREADS = 8
IQTREE_BOOTSTRAP = 1000
TOP_VAR_POSITIONS = 20                # variable positions for logo/table plots
# Reference probe IDs to EXCLUDE from crown MRCA computation
REF_IDS = {p['id'] for p in __import__('yaml').safe_load(open('config.yaml'))['reference_probes']}
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from acyltransferase.config import load_config
cfg = load_config('config.yaml')

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
# 1. Build trees for all clusters
from acyltransferase.tree import build_cluster_trees

cluster_fastas = {c: f'results/{c}_hits.faa' for c in CLUSTERS
                  if Path(f'results/{c}_hits.faa').exists()}

trees = build_cluster_trees(
    cluster_fastas = cluster_fastas,
    output_root    = 'results/trees',
    mafft_flags    = cfg.tree['mafft_flags'],
    fasttree_model = cfg.tree['fasttree_model'],
)

for name, (aln, tree) in trees.items():
    print(f'  {name}: {aln.name}  {tree.name}')

In [ ]:
# 2. IQ-TREE2 ASR (runs ~30 min per cluster on Colab T4)
import pandas as pd
from acyltransferase.tree import iqtree

iqtree_outputs = {}

if RUN_IQTREE:
    for cluster in CLUSTERS:
        aln_path = f'results/trees/{cluster}/{cluster}_aligned.faa'
        if not Path(aln_path).exists():
            print(f'  {cluster}: no alignment found, skipping')
            continue
        print(f'\nRunning IQ-TREE2 for {cluster} ...')
        iqtree_outputs[cluster] = iqtree(
            aligned_path = aln_path,
            output_dir   = f'results/iqtree/{cluster}',
            prefix       = cluster,
            model        = cfg.tree['iqtree_model'],
            bootstrap    = IQTREE_BOOTSTRAP,
            threads      = IQTREE_THREADS,
            asr          = True,
        )
        print(f'  {cluster}: done → {iqtree_outputs[cluster]["treefile"]}')

In [ ]:
# 3. Parse ASR + generate plots per cluster
from acyltransferase.asr       import map_iqtree_nodes, find_key_nodes, parse_state_file, \
                                       consensus_sequence, variable_positions, AA_ORDER
from acyltransferase.visualize import plot_sequence_logos, plot_ancestral_table, \
                                       plot_root_confidence

for cluster in CLUSTERS:
    treefile   = Path(f'results/iqtree/{cluster}/{cluster}.treefile')
    state_file = Path(f'results/iqtree/{cluster}/{cluster}.state')
    if not treefile.exists() or not state_file.exists():
        print(f'{cluster}: IQ-TREE outputs not found, skipping')
        continue

    print(f'\nParsing {cluster} ...')
    plot_dir = Path(f'results/plots/ancestral/{cluster}')
    plot_dir.mkdir(parents=True, exist_ok=True)

    # load sub-cluster assignments if available
    sub_tsv = Path(f'results/clustering/{cluster}_subcluster_assignments.tsv')
    sub_df  = pd.read_csv(sub_tsv, sep='\t') if sub_tsv.exists() else None

    # map IQ-TREE node names
    tree, node_map = map_iqtree_nodes(treefile)

    # find key nodes
    key_nodes = find_key_nodes(tree, node_map, sub_df, REF_IDS)
    key_nodes.to_csv(f'results/iqtree/{cluster}/key_nodes.tsv', sep='\t', index=False)
    print(f'  Key nodes: {len(key_nodes)}')

    # parse .state
    target_nodes = set(key_nodes['iqtree_node'].dropna())
    prob_dict    = parse_state_file(state_file, target_nodes)
    print(f'  Loaded {len(prob_dict)} node probability matrices')

    # consensus sequences
    from acyltransferase.utils import write_fasta
    node_label_map = dict(zip(key_nodes['iqtree_node'], key_nodes['node_label']))
    labelled_probs = {node_label_map.get(k, k): v for k, v in prob_dict.items()}

    write_fasta(
        [(lbl, consensus_sequence(probs)) for lbl, probs in labelled_probs.items()],
        f'results/iqtree/{cluster}/ancestral_sequences.faa',
    )

    # variable positions
    var_pos = variable_positions(labelled_probs, top_n=TOP_VAR_POSITIONS)

    # Plot 1: sequence logos
    fig = plot_sequence_logos(labelled_probs, var_pos, AA_ORDER)
    fig.savefig(plot_dir / '01_sequence_logos.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Plot 2: ancestral AA table
    fig = plot_ancestral_table(labelled_probs, var_pos, AA_ORDER)
    fig.savefig(plot_dir / '02_ancestral_table.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Plot 3: root confidence + diversity
    fig = plot_root_confidence(labelled_probs, AA_ORDER)
    fig.savefig(plot_dir / '03_root_confidence.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f'  Plots saved → {plot_dir}')

## ✓ Done
Next: **04_structures.ipynb** — AlphaFold download, ESMFold prediction, structural comparison.